In this notebook feature engineering and data cleaning are performed. The workflow is:
1. removing some features 
3. handling NaN values
2. adding feature to distinguish the single-hit event, *less than 2-hit* from multi-hit event



# 0 - General Ops

In [1]:
import pandas as pd
import numpy as np

import os
from pathlib import Path


In [2]:
# To properly visualization
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# To reset the options to their default values uncomment the following lines and run the cell
# pd.reset_option("display.max_rows")
# pd.reset_option("display.max_columns")
# pd.reset_option("display.max_colwidth")
# pd.reset_option("display.width")

In [3]:
file_path = "../../data/Engineered_Datasets"
file_name = "/bg_tgf_full.csv"
SAVE = True

In [4]:
ds = pd.read_csv(file_path + file_name, header=0)
ds.shape

(1296775, 54)

In [5]:
ds.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
event_index,1296775.0,NaN,NaN,NaN,55460.957535,53174.261141,1.0,19908.0,43670.0,70046.0,290301.0
pixel_ijk,1296775,279318,"[9,1,0]",5764,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_e0_kev,1296775.0,NaN,NaN,NaN,1079477.651033,14669401.700561,0.0,127.042,521.244,4607.5975,8247294571.619
n_voxels,1296775.0,NaN,NaN,NaN,2.635086,9.033311,1.0,1.0,1.0,2.0,371.0
E_sum,1296775.0,NaN,NaN,NaN,15320.568724,137790.838939,0.01208,116.575715,353.37273,2971.58473,15872347.0856
E_mean,1296775.0,NaN,NaN,NaN,1627.897266,3650.834602,0.01208,111.249975,277.832673,1621.630482,187533.45634
E_std,439094.0,NaN,NaN,NaN,4160.307577,9218.701831,0.00087,241.630173,1670.100546,4277.858985,361687.374836
E_min,1296775.0,NaN,NaN,NaN,815.571154,2627.447955,0.00465,71.613295,153.89547,400.704245,187533.45634
E_max,1296775.0,NaN,NaN,NaN,5851.088199,37666.738979,0.01208,114.406505,313.38868,2626.15808,4791314.74124
E_median,1296775.0,NaN,NaN,NaN,1242.326771,3193.934676,0.01208,109.71896,240.49847,875.55372,187533.45634


# 1 - Removing Features

In this section some features are removed. The reasons to remove a particular feature are:
- That feature is not available in real-case scenario: it is only available in a simulation context
- That feature is *redundant* with some other features

## Remove `event_index` feature

In [6]:
if "event_index" in ds.columns.to_list():
    ds = ds.drop(columns=["event_index"])
    print(f"Column 'event_index' removed from dataset, shape is now {ds.shape}.")
else:
    print(f"Column 'event_index' not found in dataset, shape is {ds.shape}.")

Column 'event_index' removed from dataset, shape is now (1296775, 53).


## Remove `primary_e0_kev` feature

The `primary_e0_kev` feature is **not available** in real-case scanario: it is the energy of the event at the start of the simulation (before hitting the detector). Drop event with `primary_e0_kev` value equal to zero:

In [7]:
if "primary_e0_kev" in ds.columns.to_list():
    ds = ds[ds["primary_e0_kev"] > 0]
    ds = ds.drop(columns=["primary_e0_kev"])
    print(f"Column 'primary_e0_kev' removed, shape is now {ds.shape}.")
else:
    print(f"Column 'primary_e0_kev' not found, shape is {ds.shape}.")

Column 'primary_e0_kev' removed, shape is now (1288377, 52).


## Remove *total hit*-like feature

The features `n_active`, `hits_sum`, `n_voxels` are redundant: all these features represents the number of activated sub-cubes (detector). Follow a demonstration:

In [8]:
ds[["n_voxels", "hits_sum", "n_active"]].describe(include="all").T

,count,mean,std,min,25%,50%,75%,max
n_voxels,1288377.0,2.642350,9.061870,1.0,1.0,1.0,2.0,371.0
hits_sum,1288377.0,2.809701,9.089635,1.0,1.0,1.0,2.0,371.0
n_active,1288377.0,2.642350,9.061870,1.0,1.0,1.0,2.0,371.0


In [9]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]):
    ds["hits_sum"] = ds["hits_sum"].astype(int)
    #print(f"Does the columns ['n_voxels', 'hits_sum','n_active'] have the same values? {ds["n_active"].equals(ds["n_voxels"]) and ds["n_active"].equals(ds["hits_sum"])}")
    print(f"n_voxels = hits_sum : {ds['n_voxels'].equals(ds['hits_sum'])}")
    print(f"n_voxels = n_active : {ds['n_voxels'].equals(ds['n_active'])}")
    print(f"hits_sum = n_active : {ds['hits_sum'].equals(ds['n_active'])}")

n_voxels = hits_sum : False
n_voxels = n_active : True
hits_sum = n_active : False


In [10]:
cols = ["hits_sum", "n_voxels", "n_active"]
d = ds[["particle_type"] + cols].copy()

d["hits_sum == n_voxels"] = d["hits_sum"] == d["n_voxels"]
d["n_voxels == n_active"] = d["n_voxels"] == d["n_active"]
d["all three equal"] = d["hits_sum"].eq(d["n_voxels"]) & d["n_voxels"].eq(d["n_active"])

eq_cols = ["hits_sum == n_voxels", "n_voxels == n_active", "all three equal"]

# percentage of events where the features are equal, for each particle
pct = d.groupby("particle_type")[eq_cols].mean().mul(100).round(2)
pct["n_events"] = d["particle_type"].value_counts()

# overall row
pct.loc["ALL"] = [*(d[eq_cols].mean() * 100).round(2), len(d)]

print(pct.sort_values("all three equal", ascending=False).to_string())


                          hits_sum == n_voxels  n_voxels == n_active  all three equal   n_events
particle_type                                                                                   
AlbedoPhotons                           100.00                 100.0           100.00    86814.0
AtmosphericNeutrons                     100.00                 100.0           100.00     9211.0
CosmicPhotons                           100.00                 100.0           100.00   152860.0
PrimaryAlphas                           100.00                 100.0           100.00     9079.0
PrimaryElectrons                        100.00                 100.0           100.00     7933.0
PrimaryPositrons                        100.00                 100.0           100.00    18029.0
PrimaryProtons                          100.00                 100.0           100.00    16489.0
SecondaryElectrons                      100.00                 100.0           100.00     8709.0
SecondaryPositrons            

In the background sources the above features are equals all the time, but for the tgf the ~14% of the time are different, for consistency let's drop these feature and keep only the ``hits_sum``:

Totally drop `n_active` and `n_voxels`:

In [11]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]): 
    ds = ds.drop(columns=["n_voxels", "n_active"])
    print(f"Columns 'n_voxels' and 'n_active' removed, shape is now {ds.shape}.")
else: 
    print(f"Columns 'n_voxels' and 'n_active' not found, shape is {ds.shape}.")  

Columns 'n_voxels' and 'n_active' removed, shape is now (1288377, 50).


## Remove single-valued feature

The `hits_mean_active`, `eff_E_ge_T` and `frac_active` feature contain just one unique value across avery sample in the background. For the TGFs the value are most of the time equals:

In [12]:
if "hits_mean_active" in ds.columns.to_list():
    print(ds["hits_mean_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['hits_mean_active'].nunique()}) in 'hits_mean_active': {ds['hits_mean_active'].value_counts().max()} / {ds.shape[0]}")

count    1.288377e+06
mean     1.084096e+00
std      3.474792e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.400000e+01
Name: hits_mean_active, dtype: float64

Number of same value (106) in 'hits_mean_active': 1163400 / 1288377


In [13]:
if "frac_active" in ds.columns.to_list():
    print(ds["frac_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['frac_active'].nunique()}) in 'frac_active': {ds['frac_active'].value_counts().max()} / {ds.shape[0]}")

count    1288377.0
mean           1.0
std            0.0
min            1.0
25%            1.0
50%            1.0
75%            1.0
max            1.0
Name: frac_active, dtype: float64

Number of same value (1) in 'frac_active': 1288377 / 1288377


In [14]:
if "eff_E_ge_T" in ds.columns.to_list():
    print(ds["eff_E_ge_T"].describe(include="all"))
    print(f"\nNumber of same value ({ds['eff_E_ge_T'].nunique()}) in 'eff_E_ge_T': {ds['eff_E_ge_T'].value_counts().max()} / {ds.shape[0]}")

count    1.288377e+06
mean     9.925056e-01
std      8.305216e-02
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: eff_E_ge_T, dtype: float64

Number of same value (22307) in 'eff_E_ge_T': 1257244 / 1288377


Totally drop the columns:

In [15]:
if "hits_mean_active" in ds.columns.to_list():
    ds = ds.drop(columns=["hits_mean_active"])
    print(f"Column 'hits_mean_active' removed, shape is now {ds.shape}.")
if "frac_active" in ds.columns.to_list():
    ds = ds.drop(columns=["frac_active"])
    print(f"Column 'frac_active' removed, shape is now {ds.shape}.")
if "eff_E_ge_T" in ds.columns.to_list():
    ds = ds.drop(columns=["eff_E_ge_T"])
    print(f"Column 'eff_E_ge_T' removed, shape is now {ds.shape}.")

Column 'hits_mean_active' removed, shape is now (1288377, 49).
Column 'frac_active' removed, shape is now (1288377, 48).
Column 'eff_E_ge_T' removed, shape is now (1288377, 47).


# 2 - Handling NaN features

There are feature that contains only NaN values: 

In [16]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

nan_perc_df = pd.DataFrame(
    {
        "Column": nan_counts.index, 
        "Total NaN": nan_counts.values, 
        "Percentage": (nan_counts.values / ds.shape[0] * 100).round(2)
    }
).sort_values(by="Percentage", ascending=False).reset_index(drop=True)

nan_perc_df

,Column,Total NaN,Percentage
0,f_occ,1229021,95.39
1,corr_E_hits,1200754,93.20
2,D_f_r2,1096204,85.08
3,E_skew,1063561,82.55
4,E_kurt,1063561,82.55
5,D_f,969041,75.21
6,E_std,851690,66.11
7,linearity,851690,66.11
8,planarity,851690,66.11
9,sphericity,851690,66.11


Drop those columns with 100% of NaN values:

In [17]:
columns_100perc_nan = nan_perc_df.iloc[:2]["Column"].tolist()  
ds = ds.drop(columns=columns_100perc_nan)
print(f"Columns {columns_100perc_nan} removed, shape is now {ds.shape}.")


Columns ['f_occ', 'corr_E_hits'] removed, shape is now (1288377, 45).


## Hits/Shape and NaN correlation

In [18]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

onehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 1)).sum())
twohit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 2)).sum())
treehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 3)).sum())
morethantreehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] > 3)).sum())

pd.DataFrame(
    {
        "NaN Features": nan_counts.index,
        "Percentage of NaN values": (nan_counts.values / ds.shape[0] * 100).round(2),
        "NaN with 1 hit": onehit_nan_counts.values,
        "NaN with 2 hits": twohit_nan_counts.values,
        "NaN with 3 hits": treehit_nan_counts.values,
        "NaN with >3 hits": morethantreehit_nan_counts.values 
    }
).sort_values(by=["NaN with 1 hit", "NaN with 2 hits", "NaN with 3 hits", "NaN with >3 hits"], ascending=False).reset_index(drop=True)

,NaN Features,Percentage of NaN values,NaN with 1 hit,NaN with 2 hits,NaN with 3 hits,NaN with >3 hits
0,D_f_r2,85.08,814681,206136,48611,26776
1,E_skew,82.55,814681,206136,26840,15904
2,E_kurt,82.55,814681,206136,26840,15904
3,D_f,75.21,814681,119270,24793,10297
4,E_std,66.11,814681,25975,6884,4150
5,linearity,66.11,814681,25975,6884,4150
6,planarity,66.11,814681,25975,6884,4150
7,sphericity,66.11,814681,25975,6884,4150


Most of NaN are due to the low quantity of hits per event. Let's visualize the correlation between the NaN values and the number of hits (=1, =2 or > 2)

In [19]:
one_hit = ds[ds["hits_sum"] == 1]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

print(f"Number of events with 1 hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 or less hit: {less2_hit.shape[0]} ({less2_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hit.shape[0]} ({multi_hit.shape[0]/ds.shape[0]*100:.2f}%)")

source = ds["particle_type"].unique()
less2_hit_counts = [less2_hit[less2_hit["particle_type"] == b].shape[0] for b in source]
less2_hit_percentages = [less2_hit[less2_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in source]
pd.DataFrame(
    {
        "Background": source,
        "One Hit count": [f"{one_hit[one_hit['particle_type'] == b].shape[0]} / {ds[ds['particle_type'] == b].shape[0]}" for b in source],
        "One Hit percentage": np.round([one_hit[one_hit['particle_type'] == b].shape[0]/ds[ds['particle_type'] == b].shape[0]*100 for b in source], 2),
        "Two or less Hits count": [f"{less2_hit_counts[i]} / {ds[ds['particle_type'] == source[i]].shape[0]}" for i in range(len(source))],
        #"One Hit count": f"{one_hit_counts} / {ds[ds["particle_type"] == source].shape[0]}",
        "Two or less Hits percentage": np.round(less2_hit_percentages, 2),
    }
)

Number of events with 1 hit: 814681 (63.23%)
Number of events with 2 or less hit: 1020817 (79.23%)
Number of events with multiple hits: 267560 (20.77%)


,Background,One Hit count,One Hit percentage,Two or less Hits count,Two or less Hits percentage
0,AtmosphericNeutrons,5661 / 9211,61.46,7168 / 9211,77.82
1,SecondaryProtonsUpward,5681 / 11555,49.16,6961 / 11555,60.24
2,PrimaryAlphas,1035 / 9079,11.40,1864 / 9079,20.53
3,CosmicPhotons,137476 / 152860,89.94,150935 / 152860,98.74
4,PrimaryProtons,2207 / 16489,13.38,4034 / 16489,24.46
5,AlbedoPhotons,65628 / 86814,75.60,78025 / 86814,89.88
6,PrimaryElectrons,876 / 7933,11.04,1469 / 7933,18.52
7,PrimaryPositrons,2147 / 18029,11.91,3483 / 18029,19.32
8,SecondaryElectrons,3118 / 8709,35.80,4504 / 8709,51.72
9,SecondaryProtonsDownward,5232 / 10676,49.01,6420 / 10676,60.13


⚠️ Datasets must be balanced!

In [20]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

columns_with_nan = nan_counts.index

less2_hit_nan_counts = less2_hit[columns_with_nan].isna().sum()
multi_hit_nan_counts = multi_hit[columns_with_nan].isna().sum()
less2_hit_nan_counts_percent = np.round(less2_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)
multi_hit_nan_counts_percent = np.round(multi_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)

nan_report = pd.DataFrame({
    "NaN Count": nan_counts,
    "Two or less Hits NaN Count": less2_hit_nan_counts,
    "Multi Hits NaN Count": multi_hit_nan_counts,
    "Two or less Hits NaN percentage": less2_hit_nan_counts_percent,
    "Multi Hits NaN percentage": multi_hit_nan_counts_percent
})

nan_report.index.name = "Column"
nan_report.reset_index()

,Column,NaN Count,Two or less Hits NaN Count,Multi Hits NaN Count,Two or less Hits NaN percentage,Multi Hits NaN percentage
0,E_std,851690,840656,11034,98.70,1.30
1,E_skew,1063561,1020817,42744,95.98,4.02
2,E_kurt,1063561,1020817,42744,95.98,4.02
3,linearity,851690,840656,11034,98.70,1.30
4,planarity,851690,840656,11034,98.70,1.30
5,sphericity,851690,840656,11034,98.70,1.30
6,D_f,969041,933951,35090,96.38,3.62
7,D_f_r2,1096204,1020817,75387,93.12,6.88


The geometrical features and some energy features are obviously correlated with the number of hits: to define some of these features, at least 2 hits are required (e.g., standard deviation). In particular:
- E_std needs at least 2 hits,
- linearity, planarity and sphericity need at least 2,3+ hits
- E_skew (skewness) at lest 3 hits
- E_kurt 

Two different approach are possible: 
- setting those feature to a default value (e.g., 0) if the number of hits is equal to 1 (or <=2 or = 2)
- create differents sub-datasets:
    - One-hit dataset which does not contain the NaN features
    - 2-hit dataset 
    - *"Less than 2"*-hit dataset
    - Multi-hit dataset and handle separately the remaining NaNs 

# 3 - Distinguish events by number of hit

In [21]:
one_hit = ds[ds["hits_sum"] == 1]
two_hit = ds[ds["hits_sum"] == 2]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

print(f"Number of events with 1 hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 hits: {two_hit.shape[0]} ({two_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 or less hit: {less2_hit.shape[0]} ({less2_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hit.shape[0]} ({multi_hit.shape[0]/ds.shape[0]*100:.2f}%)")

source = ds["particle_type"].unique()
one_hit_counts = [one_hit[one_hit["particle_type"] == b].shape[0] for b in source]
one_hit_percentages = [one_hit[one_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in source]
two_hit_counts = [two_hit[two_hit["particle_type"] == b].shape[0] for b in source]
two_hit_percentages = [two_hit[two_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in source]
less2_hit_counts = [less2_hit[less2_hit["particle_type"] == b].shape[0] for b in source]
less2_hit_percentages = [less2_hit[less2_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in source]
multi_hit_counts = [multi_hit[multi_hit["particle_type"] == b].shape[0] for b in source]
multi_hit_percentages = [multi_hit[multi_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in source]
pd.DataFrame(
    {
        "Background": source,
        "One Hit count": [f"{one_hit_counts[i]} / {ds[ds['particle_type'] == source[i]].shape[0]}" for i in range(len(source))],
        "One Hit percentage": np.round(one_hit_percentages, 2),
        "Two Hits count": [f"{two_hit_counts[i]} / {ds[ds['particle_type'] == source[i]].shape[0]}" for i in range(len(source))],
        "Two Hits percentage": np.round(two_hit_percentages, 2),
        "Two or less Hits count": [f"{less2_hit_counts[i]} / {ds[ds['particle_type'] == source[i]].shape[0]}" for i in range(len(source))],
        "Two or less Hits percentage": np.round(less2_hit_percentages, 2),
        "Multiple Hits count": [f"{multi_hit_counts[i]} / {ds[ds['particle_type'] == source[i]].shape[0]}" for i in range(len(source))],
        "Multiple Hits percentage": np.round(multi_hit_percentages, 2)
    }
).sort_values(by="Two or less Hits percentage", ascending=False)

Number of events with 1 hit: 814681 (63.23%)
Number of events with 2 hits: 206136 (16.00%)
Number of events with 2 or less hit: 1020817 (79.23%)
Number of events with multiple hits: 267560 (20.77%)


,Background,One Hit count,One Hit percentage,Two Hits count,Two Hits percentage,Two or less Hits count,Two or less Hits percentage,Multiple Hits count,Multiple Hits percentage
3,CosmicPhotons,137476 / 152860,89.94,13459 / 152860,8.80,150935 / 152860,98.74,1925 / 152860,1.26
5,AlbedoPhotons,65628 / 86814,75.60,12397 / 86814,14.28,78025 / 86814,89.88,8789 / 86814,10.12
11,TGF,578647 / 928000,62.35,165426 / 928000,17.83,744073 / 928000,80.18,183927 / 928000,19.82
0,AtmosphericNeutrons,5661 / 9211,61.46,1507 / 9211,16.36,7168 / 9211,77.82,2043 / 9211,22.18
1,SecondaryProtonsUpward,5681 / 11555,49.16,1280 / 11555,11.08,6961 / 11555,60.24,4594 / 11555,39.76
9,SecondaryProtonsDownward,5232 / 10676,49.01,1188 / 10676,11.13,6420 / 10676,60.13,4256 / 10676,39.87
8,SecondaryElectrons,3118 / 8709,35.80,1386 / 8709,15.91,4504 / 8709,51.72,4205 / 8709,48.28
10,SecondaryPositrons,6973 / 29022,24.03,4908 / 29022,16.91,11881 / 29022,40.94,17141 / 29022,59.06
4,PrimaryProtons,2207 / 16489,13.38,1827 / 16489,11.08,4034 / 16489,24.46,12455 / 16489,75.54
2,PrimaryAlphas,1035 / 9079,11.40,829 / 9079,9.13,1864 / 9079,20.53,7215 / 9079,79.47


Two possible approach:
- mantain one single dataset with a *flag*-like feature which distinguish between single or multi hits 
- divide the dataset in sub-datasets

NaN values must be handled!

In [22]:
one_hit = ds[ds["hits_sum"] == 1]
two_hit = ds[ds["hits_sum"] == 2]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

### One-hit Dataset (and *"2 or less"*-hit)

In [23]:
print(f"\"Raw\" One hit dataset shape: {one_hit.shape}")
print(f"\nNaN values in one hit dataset:")
oh_nan_val = one_hit.isna().sum()
oh_nan_val = oh_nan_val[oh_nan_val > 0]
print(oh_nan_val)

"Raw" One hit dataset shape: (814681, 45)

NaN values in one hit dataset:
E_std         814681
E_skew        814681
E_kurt        814681
linearity     814681
planarity     814681
sphericity    814681
D_f           814681
D_f_r2        814681
dtype: int64


In [24]:
print(f"\"Raw\" Less 2 hit dataset shape: {less2_hit.shape}")
print(f"\nNaN values in less 2 hit dataset:")
l2_nan_val = less2_hit.isna().sum()
l2_nan_val = l2_nan_val[l2_nan_val > 0]
print(l2_nan_val)

"Raw" Less 2 hit dataset shape: (1020817, 45)

NaN values in less 2 hit dataset:
E_std          840656
E_skew        1020817
E_kurt        1020817
linearity      840656
planarity      840656
sphericity     840656
D_f            933951
D_f_r2        1020817
dtype: int64


Just simply drop this features

In [25]:
## less one hit dataset cleaning
columns_to_drop = oh_nan_val.index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with NaN values in one hit dataset: {columns_to_drop}")
    one_hit = one_hit.drop(columns=columns_to_drop)

    print(f"NaN values in one hit dataset after dropping NaN values: {one_hit.isna().sum().sum()}")

print(f'"Cleaned" One hit dataset shape: {one_hit.shape}')

## less 2 hit dataset cleaning
columns_to_drop = l2_nan_val.index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with NaN values in less 2 hit dataset: {columns_to_drop}")
    less2_hit = less2_hit.drop(columns=columns_to_drop)

    print(f"NaN values in less 2 hit dataset after dropping NaN values: {less2_hit.isna().sum().sum()}")

print(f'"Cleaned" Less 2 hit dataset shape: {less2_hit.shape}')


Dropping columns with NaN values in one hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in one hit dataset after dropping NaN values: 0
"Cleaned" One hit dataset shape: (814681, 37)

Dropping columns with NaN values in less 2 hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in less 2 hit dataset after dropping NaN values: 0
"Cleaned" Less 2 hit dataset shape: (1020817, 37)


In [26]:
one_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,814681,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.000000,814681.0,814681.0,814681.0,8.146810e+05,814681.0,814681.0,814681.000000,814681.000000,814681.000000,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681.0,814681
unique,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12
top,"[9,1,0]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TGF
freq,5658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,578647
mean,NaN,830.384800,830.384800,830.384800,830.384800,830.384800,830.384800,830.384800,830.384800,830.384800,830.384800,830.384800,0.0,0.0,0.0,3.143420e-17,1.0,1.0,4.489890,4.483683,3.745209,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
std,NaN,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,2796.294561,0.0,0.0,0.0,7.740603e-17,0.0,0.0,3.260405,3.204588,3.221812,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
min,NaN,0.012080,0.012080,0.012080,0.012080,0.012080,0.012080,0.012080,0.012080,0.012080,0.012080,0.012080,0.0,0.0,-0.0,0.000000e+00,1.0,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
25%,NaN,86.926510,86.926510,86.926510,86.926510,86.926510,86.926510,86.926510,86.926510,86.926510,86.926510,86.926510,0.0,0.0,0.0,0.000000e+00,1.0,1.0,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
50%,NaN,152.887350,152.887350,152.887350,152.887350,152.887350,152.887350,152.887350,152.887350,152.887350,152.887350,152.887350,0.0,0.0,0.0,0.000000e+00,1.0,1.0,4.000000,5.000000,3.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
75%,NaN,422.135130,422.135130,422.135130,422.135130,422.135130,422.135130,422.135130,422.135130,422.135130,422.135130,422.135130,0.0,0.0,0.0,0.000000e+00,1.0,1.0,8.000000,8.000000,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN


In [27]:
less2_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,1020817,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1020817.0,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1.020817e+06,1020817.0,1.020817e+06,1020817
unique,51474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12
top,"[9,1,0]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TGF
freq,5732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,744073
mean,NaN,1.291792e+03,1.037139e+03,8.747519e+02,1.199527e+03,1.037139e+03,9.072294e+02,9.559456e+02,1.118333e+03,1.167049e+03,1.183288e+03,1.196279e+03,1.623874e+02,1.623874e+02,8.190754e-02,2.430232e-02,9.603611e-01,1.0,4.488779e+00,4.469248e+00,3.833899e+00,1.567085e-01,2.292814e-01,7.382912e-18,1.687702e-19,-2.538325e-04,-2.272434e-04,7.925136e-03,1.937713e-01,1.942571e-01,2.126130e-01,1.201932e+00,9.139248e-02,1.116316e-01,0.0,4.275846e-01,NaN
std,NaN,3.912359e+03,2.988140e+03,2.778066e+03,3.434148e+03,2.988140e+03,2.797797e+03,2.848979e+03,3.186619e+03,3.329902e+03,3.381205e+03,3.423432e+03,9.091379e+02,9.091379e+02,1.958652e-01,6.093135e-02,1.056353e-01,0.0,3.193978e+00,3.139972e+00,3.160613e+00,4.524644e-01,1.150458e+00,8.072227e-17,5.574530e-18,2.110035e-01,2.117970e-01,2.164888e-01,6.788630e-01,6.762341e-01,7.342221e-01,4.014422e-01,2.881666e-01,3.608212e-01,0.0,1.181054e+00,NaN
min,NaN,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,1.208000e-02,0.000000e+00,0.000000e+00,-0.000000e+00,0.000000e+00,5.000011e-01,1.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-9.998728e-01,-9.999230e-01,-9.999657e-01,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,NaN
25%,NaN,1.019708e+02,9.708611e+01,8.286492e+01,1.002819e+02,9.708611e+01,8.856147e+01,9.251909e+01,9.934717e+01,1.000911e+02,1.001905e+02,1.002622e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.0,1.221362e+00,1.211327e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,NaN
50%,NaN,1.977405e+02,1.820084e+02,1.616537e+02,1.910073e+02,1.820084e+02,1.709622e+02,1.757040e+02,1.871040e+02,1.896399e+02,1.903711e+02,1.908625e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.0,4.300813e+00,4.683014e+00,3.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,NaN
75%,NaN,8.534438e+02,6.806785e+02,4.611761e+02,7.715959e+02,6.806785e+02,5.360585e+02,6.064704e+02,7.297114e+02,7.557762e+02,7.637386e+02,7.700637e+02,0.000000e+00,0.000000e+00,0.000000e+00,2.220446e-16,1.000000e+00,1.0,7.939928e+00,7.050224e+00,7.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,NaN


Almost every faeture is meaningless if the number of hits is equal to 1 (or <=2). Drop those feature:

In [28]:
one_hit = one_hit[["pixel_ijk", "E_sum", "com_x", "com_y", "com_z", "hits_sum", "particle_type"]]
less2_hit = less2_hit[["pixel_ijk", "E_sum", "com_x", "com_y", "com_z", "hits_sum", "particle_type"]]

### Two-hit Dataset

In [29]:
print(f"\"Raw\" Two hit dataset shape: {two_hit.shape}")
print(f"\nNaN values in two hit dataset:")
th_nan_val = two_hit.isna().sum()
th_nan_val = th_nan_val[th_nan_val > 0]
print(th_nan_val)

"Raw" Two hit dataset shape: (206136, 45)

NaN values in two hit dataset:
E_std          25975
E_skew        206136
E_kurt        206136
linearity      25975
planarity      25975
sphericity     25975
D_f           119270
D_f_r2        206136
dtype: int64


Just simply drop full nan features and impute 0 to `D_f`:

In [30]:
columns_to_drop = th_nan_val[th_nan_val == two_hit.shape[0]].index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with all NaN values in two hit dataset: {columns_to_drop}")
    two_hit = two_hit.drop(columns=columns_to_drop)

    print(f"NaN values in two hit dataset after dropping all NaN values: {two_hit.isna().sum().sum()}")

## impute 0 to D_f 
if "D_f" in two_hit.columns:
    two_hit["D_f"] = two_hit["D_f"].fillna(0)
    print(f"NaN values in two hit dataset after imputing 0 to D_f: {two_hit.isna().sum().sum()}")


Dropping columns with all NaN values in two hit dataset: ['E_skew', 'E_kurt', 'D_f_r2']
NaN values in two hit dataset after dropping all NaN values: 223170
NaN values in two hit dataset after imputing 0 to D_f: 103900


In [31]:
two_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_std,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,linearity,planarity,sphericity,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,D_f,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,206136,206136.000000,206136.000000,180161.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.0,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,2.061360e+05,2.061360e+05,1.801610e+05,1.801610e+05,1.801610e+05,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.000000,206136.0,2.061360e+05,206136.000000,206136.000000,206136.0,206136.000000,206136
unique,51474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12
top,"[0,9,0];[0,8,0]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TGF
freq,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165426
mean,NaN,3115.345576,1854.264692,1301.231272,1050.097347,2658.432037,1854.264692,1210.930816,1452.181019,2256.348364,2497.598568,2578.015302,2642.348690,804.167345,804.167345,0.405619,0.120349,0.803702,1.0,4.484387,4.412201,4.184413,0.776044,1.135436,3.656131e-17,8.357757e-19,1.000000e+00,1.776453e-17,3.889354e-19,-0.001257,-0.001125,0.039246,0.959585,0.961991,1.052892,2.0,9.765081e-17,0.452590,0.552817,0.0,2.117464,NaN
std,NaN,6382.076293,3532.444650,2823.498731,2697.695803,4983.253980,3532.444650,2783.048501,2997.266649,4214.408162,4667.929416,4824.460837,4951.324814,1891.302787,1891.302787,0.242233,0.082622,0.156552,0.0,2.916682,2.869697,2.879560,0.730201,2.350656,1.766408e-16,1.238278e-17,3.591367e-17,3.613996e-17,3.264697e-18,0.469555,0.471321,0.480485,1.243932,1.235328,1.336001,0.0,1.203691e-16,0.497748,0.633117,0.0,1.824678,NaN
min,NaN,1.006150,1.006150,0.000870,0.016470,1.006150,1.006150,1.006150,1.006150,1.006150,1.006150,1.006150,1.006150,0.000000,0.000000,-0.000000,0.000000,0.500001,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,-0.999873,-0.999923,-0.999966,0.000000,0.000000,0.000000,2.0,-9.309075e-17,0.000000,0.000000,0.0,0.000000,NaN
25%,NaN,361.742692,182.880265,72.656693,54.258533,259.240325,182.880265,101.636622,137.796906,222.210633,244.719154,252.253139,257.807554,21.205027,21.205027,0.199764,0.043019,0.668619,1.0,2.000000,1.989121,1.653734,0.333247,0.111054,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,-0.138847,-0.148927,-0.051534,0.000000,0.000000,0.000000,2.0,0.000000e+00,0.000000,0.000000,0.0,1.000000,NaN
50%,NaN,1156.473605,608.804237,280.002300,216.118190,886.492525,608.804237,335.405087,450.167736,751.519996,832.886083,859.998051,881.020884,133.394175,133.394175,0.460798,0.125018,0.826881,1.0,4.371755,4.344935,4.000000,0.553596,0.306468,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,2.0,0.000000e+00,0.000000,0.000000,0.0,1.414214,NaN
75%,NaN,3716.900840,2143.955260,1433.802450,510.998910,3344.236510,2143.955260,873.079841,1400.029552,2782.744509,3126.087644,3235.104102,3323.997337,798.885353,798.885352,0.635152,0.197227,0.949575,1.0,7.000000,7.000000,6.690819,1.047144,1.096510,9.658691e-18,0.000000e+00,1.000000e+00,2.004095e-17,0.000000e+00,0.134328,0.137594,0.294347,1.000000,1.000000,1.000000,2.0,2.402570e-16,1.000000,1.000000,0.0,2.828427,NaN


Almost every faeture is meaningless if the number of hits is equal to 2. Drop those feature:

In [32]:
two_hit = two_hit[["pixel_ijk", "E_sum", "E_min", "E_max", "com_x", "com_y", "com_z", "rg", "span_x", "span_y", "span_z", "D_f", "hits_sum", "particle_type"]]

### Multi-hit Dataset

In [33]:
print(f"\"Raw\" Multi hits dataset shape: {multi_hit.shape}")
print(f"\n\nNaN values in multi hits dataset:")
mh_nan_val = multi_hit.isna().sum()
mh_nan_val = mh_nan_val[mh_nan_val > 0]
print(mh_nan_val)

"Raw" Multi hits dataset shape: (267560, 45)


NaN values in multi hits dataset:
E_std         11034
E_skew        42744
E_kurt        42744
linearity     11034
planarity     11034
sphericity    11034
D_f           35090
D_f_r2        75387
dtype: int64


<!-- To compute the skewness and the kurtosis coeff. at least 3 hits are needed. To avoid to create a third dataset, and since there are almost ~40% missing values, we can decide to totally drop these features. -->

R2 index may be dropped. For the Fractal Dimension we decide to impute a default value 0, preserving the meaning of the feature: a too small to be computed fractal dimension may be set to 0.  


In [34]:
# mh_cols_drop = ["E_skew", "E_kurt", "D_f_r2"]
mh_cols_drop = ["D_f_r2"]

if all(col in multi_hit.columns.to_list() for col in mh_cols_drop):
    print(f"\nDropping columns with NaN values in multi hits dataset: {mh_cols_drop}")
    multi_hit = multi_hit.drop(columns=mh_cols_drop)

    print(f"NaN values in multi hits dataset after dropping NaN values: {multi_hit.isna().sum().sum()}")

# impute 0 to D_f
if "D_f" in multi_hit.columns.to_list():
    multi_hit["D_f"] = multi_hit["D_f"].fillna(0)
    print(f"NaN values in 'D_f' column after imputation: {multi_hit['D_f'].isna().sum()}")

print(f'"Cleaned" Multi hits dataset shape: {multi_hit.shape}')


Dropping columns with NaN values in multi hits dataset: ['D_f_r2']
NaN values in multi hits dataset after dropping NaN values: 164714
NaN values in 'D_f' column after imputation: 0
"Cleaned" Multi hits dataset shape: (267560, 44)


Drop rows with NaN values:

In [35]:
print(multi_hit.shape)
print(multi_hit["particle_type"].value_counts())

print("="*50)
multi_hit = multi_hit.dropna()
print(multi_hit.shape)
print(multi_hit["particle_type"].value_counts())

(267560, 44)
particle_type
TGF                         183927
SecondaryPositrons           17141
PrimaryPositrons             14546
PrimaryProtons               12455
AlbedoPhotons                 8789
PrimaryAlphas                 7215
PrimaryElectrons              6464
SecondaryProtonsUpward        4594
SecondaryProtonsDownward      4256
SecondaryElectrons            4205
AtmosphericNeutrons           2043
CosmicPhotons                 1925
Name: count, dtype: int64
(224816, 44)
particle_type
TGF                         141183
SecondaryPositrons           17141
PrimaryPositrons             14546
PrimaryProtons               12455
AlbedoPhotons                 8789
PrimaryAlphas                 7215
PrimaryElectrons              6464
SecondaryProtonsUpward        4594
SecondaryProtonsDownward      4256
SecondaryElectrons            4205
AtmosphericNeutrons           2043
CosmicPhotons                 1925
Name: count, dtype: int64


### Summary and saving 

In [36]:
print(f"Final shape of one hit dataset: {one_hit.shape}")
print(f"Final shape of two hit dataset: {two_hit.shape}")
print(f"Final shape of less 2 hit dataset: {less2_hit.shape}")
print(f"Final shape of multi hits dataset: {multi_hit.shape}")

Final shape of one hit dataset: (814681, 7)
Final shape of two hit dataset: (206136, 14)
Final shape of less 2 hit dataset: (1020817, 7)
Final shape of multi hits dataset: (224816, 44)


In [37]:
if "multi_hit" in one_hit.columns.to_list():
    one_hit = one_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from one_hit dataset, shape is now {one_hit.shape}.")
if "multi_hit" in two_hit.columns.to_list():
    two_hit = two_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from two_hit dataset, shape is now {two_hit.shape}.")
if "multi_hit" in less2_hit.columns.to_list():
    less2_hit = less2_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from less2_hit dataset, shape is now {less2_hit.shape}.")
if "multi_hit" in multi_hit.columns.to_list():
    multi_hit = multi_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from multi_hit dataset, shape is now {multi_hit.shape}.")

In [38]:
if SAVE:
    one_hit.to_csv(file_path + "/bgtgf_1hit_dataset.csv", index=False)
    two_hit.to_csv(file_path + "/bgtgf_2hit_dataset.csv", index=False)
    less2_hit.to_csv(file_path + "/bgtgf_le2hit_dataset.csv", index=False) # NAME -> background terrestrial gamma-ray flash (bgtgf) less or equal to 2 
    multi_hit.to_csv(file_path + "/bgtgf_ge3hit_dataset.csv", index=False) # NAME -> background terrestrial gamma-ray flash (bgtgf) greater or equal to 3
    print(f"Datasets saved to {file_path}.")

Datasets saved to ../../data/Engineered_Datasets.
